# Make the fire/smoke model for the ESP32-S3 CAM (free Google Colab GPU)

Run the cells **from top to bottom** (click each one, then press **Shift + Enter**). At the end you download `fire_det_s8_esp32s3.espdl`, the file the ESP32 firmware needs.

**Before you start:**
1. **Runtime → Change runtime type → T4 GPU → Save.**
2. Upload the `fire_yolo_esp32s3` folder to **My Drive** in Google Drive.
3. Upload your dataset zip(s) to **My Drive** too: Roboflow Universe downloads in **YOLOv11** format. Optionally, add a zip of photos with no fire (for example, DeepQuestAI's Fire-Smoke-Dataset).

Progress is saved to your Drive after every step. If Colab disconnects, run cells 1–3 again and continue from the step you were on.

In [ ]:
# 1) Check that a GPU is attached (you should see "Tesla T4" or similar)
!nvidia-smi -L || echo "NO GPU: Runtime > Change runtime type > T4 GPU, then run this cell again"

In [ ]:
# 2) Connect Google Drive and set where your files are  (edit the two paths if yours differ)
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/fire_yolo_esp32s3'   # the project folder you uploaded
# One or more datasets WITH boxes (Roboflow "YOLOv11" zips). They are merged.
DATASET_ZIPS = ['/content/drive/MyDrive/fire-smoke.zip']
# Optional: photos WITHOUT fire/smoke, e.g. DeepQuestAI's Fire-Smoke-Dataset zip (only its
# "Neutral" folder is used) or a folder of your own room photos. Leave [] if you have none.
NEGATIVES = []

import os
assert os.path.isdir(PROJECT), f'Project folder not found: {PROJECT}'
for p in DATASET_ZIPS + NEGATIVES:
    assert os.path.exists(p), f'Not found: {p}'

# Work on Colab's fast local disk; results are copied back to Drive after each step
!mkdir -p /content/work
!rsync -a --exclude datasets "{PROJECT}/" /content/work/
%cd /content/work
def save_to_drive():
    !rsync -a runs export field_data "{PROJECT}/" 2>/dev/null; echo "Saved progress to Drive"

In [ ]:
# 3) Install the tools (about 2 minutes)
!pip install -q -r requirements.txt

# Steps 4-5 use Espressif's quantizer, which needs Python 3.12 (Colab now runs 3.13),
# so they get their own small Python 3.12 environment, made the first time it's needed.
EXPORT_PY = '/content/export_env/bin/python'
def export_env_ok():
    import subprocess
    test = f"{EXPORT_PY} -c 'import torch, torchvision, esp_ppq, ultralytics, onnx, onnxruntime'"
    return subprocess.run(test, shell=True, capture_output=True).returncode == 0

def setup_export_env():
    if export_env_ok():
        return
    print('Setting up Python 3.12 for export (3-6 minutes, once per session)...')
    !pip install -q uv
    !rm -rf /content/export_env
    !uv venv --python 3.12 /content/export_env
    # --python: Colab sets UV_SYSTEM_PYTHON, which would otherwise install into its own Python 3.13
    !uv pip install --python /content/export_env/bin/python -r requirements-export.txt
    if not export_env_ok():
        raise RuntimeError('The export tools did not install. Scroll up for the red error and send it to Claude.')
    print('Export environment ready.')


In [ ]:
# 4) Prepare the dataset: merges your datasets, maps their class names to 0 = fire, 1 = smoke,
#    adds the no-fire photos, splits train/val
args = ' '.join(f'"{p}"' for p in DATASET_ZIPS)
if NEGATIVES:
    args += ' --negatives ' + ' '.join(f'"{p}"' for p in NEGATIVES)
!python 0_prepare_dataset.py {args}

## Training length

With `QUICK_TRY = True`, all training finishes in roughly 1–2 hours on a T4 for a few thousand images, so you can check that everything works. For the best model, set it to `False` and expect several hours. You may need to reconnect and continue.

In [ ]:
QUICK_TRY = True

TEACHER_EPOCHS  = 30 if QUICK_TRY else 100
STUDENT_EPOCHS  = 60 if QUICK_TRY else 200
FINETUNE_EPOCHS = 20 if QUICK_TRY else 60
print(TEACHER_EPOCHS, STUDENT_EPOCHS, FINETUNE_EPOCHS)

In [ ]:
# Step 1: train the big "teacher" model (yolo11s). Watch mAP50 go up in the table.
!python 1_train_teacher.py --epochs {TEACHER_EPOCHS}
save_to_drive()

In [ ]:
# Step 2: train the small "student" (fire-slim) while it copies the teacher
!python 2_train_student.py --epochs {STUDENT_EPOCHS}
save_to_drive()

In [ ]:
# Step 3: prune (remove ~half the model) and fine-tune to win the accuracy back
!python 3_prune.py --epochs {FINETUNE_EPOCHS}
save_to_drive()

In [ ]:
# Step 4: export in the layout the ESP32 needs (runs in the Python 3.12 environment)
setup_export_env()
!{EXPORT_PY} 4_export_onnx.py
save_to_drive()

In [ ]:
# Step 5: convert to int8 for the ESP32-S3. At the end it prints float vs int8 accuracy.
setup_export_env()
!{EXPORT_PY} 5_quantize_espdl.py
save_to_drive()

In [ ]:
# 6) Download the model -> put it in the firmware as main/models/fire_det.espdl
from google.colab import files
files.download('export/fire_det_s8_esp32s3.espdl')

## How good is it?

- **mAP50** printed after steps 1, 2, 3 and 5 is the main score (0–1, higher is better). Around 0.6 or more is a decent start for fire/smoke datasets.
- Step 5 shows **float32 vs int8**. They should be within a few points of each other.
- Check that the student and pruned scores didn't drop far below the teacher's. If they did, train longer (`QUICK_TRY = False`) or prune less: `!python 3_prune.py --ratio 0.2`.
- To see the teacher's predictions on validation images, run the cell below.

In [ ]:
from IPython.display import Image, display
display(Image('runs/1_teacher/val_batch0_pred.jpg', width=900))

---
# Later: improve the model with your fire cam feedback (steps 6–7)

After testing on the device, where B2 = right and B3 = wrong saves samples:
1. On your PC, copy the samples off the board: press B2 on the fire cam, then run `python pull_field_data.py --wifi <address shown on the OLED>`. You can also use `--port COM5` over the USB cable. This creates a `field_data` folder.
2. Upload that `field_data` folder into the `fire_yolo_esp32s3` folder on Drive.
3. In a new Colab session, run cells **1–4** again. Cell 4 rebuilds the original dataset. Then run the cells below.

In [ ]:
# Step 6: turn your right/wrong verdicts into training data.
# If you labeled the "missed" photos in Roboflow, add:  --labeled "/content/drive/MyDrive/field-labels.zip"
!python 6_add_field_data.py
save_to_drive()

In [ ]:
# Step 7: fine-tune your model on it (prints false alarms / hits on your own photos, before and after)
!python 7_finetune_field.py
save_to_drive()

In [ ]:
# Export + int8 again, then download the improved model
setup_export_env()
!{EXPORT_PY} 4_export_onnx.py --weights runs/7_field/weights/best.pt
!{EXPORT_PY} 5_quantize_espdl.py --data data/fire_field.yaml
save_to_drive()
from google.colab import files
files.download('export/fire_det_s8_esp32s3.espdl')